# Density & Coverage: Post-trained vs Base Model Trajectories

Measures how post-trained model trajectories concentrate relative to base model
trajectories in heuristic frequency space, using the k-NN sphere metric
(Naeem et al.) with k=3.

Produces the paper table:
- **Post-trained pairs**: Qwen3-1.7B-GRPO, Olmo-3-7B-Think-RL-Zero, Olmo-3-7B-Think-RLVR
- **Cross-model baseline**: Olmo-3-7B (real) vs Qwen3-1.7B-Base (fake)

In [1]:
import json
import os
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import pairwise_distances

_NB_DIR = Path(os.path.abspath(''))
HEUR_DATA_DIR = (_NB_DIR / '../data/llm-label-heuristics').resolve()
SEM_TRAIN_DATA_DIR = (_NB_DIR / '../data/semantic_space_training').resolve()
OUTPUT_DIR = _NB_DIR / 'outputs/density_coverage_split'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELER = 'Qwen-Qwen3.5-27B'
K = 3

MODEL_REGISTRY = {
    'baseline': 'base_Qwen-Qwen3-1.7B-Base_global_step_0',
    'grpo': 'GRPO-Qwen3-1.7B-Base-hendrycks_math-20260425_010901_global_step_200',
    'olmo_base': 'olmo-3-7b',
    'olmo_rl_zero': 'olmo-3-7b-rl-zero',
    'olmo_rlvr': 'olmo-3-7b-think-rlvr',
}

MODEL_SOURCE = {
    'baseline': 'semantic_space_training',
    'grpo': 'semantic_space_training',
    'olmo_base': 'heuristics',
    'olmo_rl_zero': 'heuristics',
    'olmo_rlvr': 'heuristics',
}

# (base_key, target_key, label, section)
PAPER_PAIRS = [
    ('baseline',  'grpo',         'Qwen3-1.7B-Base → Qwen3-1.7B-GRPO',      'Post-trained'),
    ('olmo_base', 'olmo_rl_zero', 'Olmo-3-7B → Olmo-3-7B-Think-RL-Zero',    'Post-trained'),
    ('olmo_base', 'olmo_rlvr',    'Olmo-3-7B → Olmo-3-7B-Think-RLVR',       'Post-trained'),
    ('olmo_base', 'baseline',     'Olmo-3-7B → Qwen3-1.7B-Base',            'Cross-model baseline'),
]

print('HEUR_DATA_DIR:', HEUR_DATA_DIR, '| exists:', HEUR_DATA_DIR.exists())
print('SEM_TRAIN_DATA_DIR:', SEM_TRAIN_DATA_DIR, '| exists:', SEM_TRAIN_DATA_DIR.exists())

HEUR_DATA_DIR: /data3/jongsong/SHAPE/data/llm-label-heuristics | exists: True
SEM_TRAIN_DATA_DIR: /data3/jongsong/SHAPE/data/semantic_space_training | exists: True


In [2]:
def load_sample_index(model_name: str, perturb: str, result: str, source: str = 'heuristics') -> dict:
    if source == 'semantic_space_training':
        dir_path = SEM_TRAIN_DATA_DIR / f'math_perturb_{perturb}' / f'{model_name}_results_{result}' / LABELER
    else:
        dir_path = HEUR_DATA_DIR / f'{model_name}_math_perturb_{perturb}_result_{result}' / LABELER
    if not dir_path.exists():
        raise FileNotFoundError(f'Directory not found: {dir_path}')
    sample_index = defaultdict(dict)
    pattern = re.compile(r'^(\d+)_(\d+)\.json$')
    for json_file in sorted(dir_path.glob('*.json')):
        m = pattern.match(json_file.name)
        if not m:
            continue
        prob_id = int(m.group(1))
        sample_idx = int(m.group(2))
        with open(json_file, 'r') as f:
            chunks = json.load(f)
        sample_index[prob_id][sample_idx] = chunks
    return dict(sample_index)


def get_parent_category(tag: str) -> 'str | None':
    if re.match(r'H\d+[a-z]$', tag):
        return re.match(r'(H\d+)[a-z]$', tag).group(1)
    if re.match(r'N\d+', tag):
        return 'N'
    return None


def extract_tags(chunk: dict) -> list:
    raw = chunk.get('ontology_tag', [])
    if isinstance(raw, str):
        raw = [raw]
    tags = set(raw)
    parents = {get_parent_category(t) for t in tags} - {None}
    return sorted(tags | parents)


def extract_features(chunks: list) -> dict:
    heuristic_dist = Counter()
    for chunk in chunks:
        heuristic_dist.update(extract_tags(chunk))
    return {'heuristic_dist': heuristic_dist}


def h_vec(feat: dict, h_tags: list) -> np.ndarray:
    dist = feat['heuristic_dist']
    vec = np.array([float(dist.get(h, 0)) for h in h_tags])
    s = vec.sum()
    return vec / s if s > 0 else vec


PERTURBATIONS = ['original', 'simple', 'hard']
FEATURE_STORE = {}
count_rows = []

for model_key, model_name in MODEL_REGISTRY.items():
    for perturb in PERTURBATIONS:
        label = f'{model_key}_{perturb}_correct'
        try:
            source = MODEL_SOURCE.get(model_key, 'heuristics')
            prob_map = load_sample_index(model_name, perturb, 'correct', source=source)
        except FileNotFoundError as e:
            print('[WARN]', e)
            continue
        FEATURE_STORE[label] = {
            prob_id: {idx: extract_features(chunks) for idx, chunks in sample_map.items()}
            for prob_id, sample_map in prob_map.items()
        }
        count_rows.append({
            'label': label,
            'n_problems': len(prob_map),
            'n_samples': sum(len(v) for v in prob_map.values()),
        })

count_df = pd.DataFrame(count_rows).sort_values('label').reset_index(drop=True)
display(count_df)

,label,n_problems,n_samples
0,baseline_hard_correct,56,215
1,baseline_original_correct,79,349
2,baseline_simple_correct,80,338
3,grpo_hard_correct,63,253
4,grpo_original_correct,77,351
5,grpo_simple_correct,82,371
6,olmo_base_hard_correct,103,463
7,olmo_base_original_correct,111,535
8,olmo_base_simple_correct,111,525
9,olmo_rl_zero_hard_correct,108,522


In [3]:
all_h_tags = sorted({
    h
    for prob_map in FEATURE_STORE.values()
    for idx_feat in prob_map.values()
    for feat in idx_feat.values()
    for h in feat['heuristic_dist'].keys()
})

print(f'h_tags vocabulary size: {len(all_h_tags)}')
print(f'Sample: {all_h_tags[:12]}')

h_tags vocabulary size: 31
Sample: ['E', 'H1', 'H10', 'H11', 'H11a', 'H11b', 'H11c', 'H11d', 'H11e', 'H11f', 'H2', 'H3']


In [4]:
def compute_density_coverage(real_vecs, fake_vecs, k=5, metric='cosine'):
    N, M = len(real_vecs), len(fake_vecs)
    if N < 2 or M == 0:
        return np.nan, np.nan
    k_eff = min(k, N - 1)
    real_real_dists = pairwise_distances(real_vecs, metric=metric)
    np.fill_diagonal(real_real_dists, np.inf)
    knn_dists = np.sort(real_real_dists, axis=1)[:, k_eff - 1]
    real_fake_dists = pairwise_distances(real_vecs, fake_vecs, metric=metric)
    contained = real_fake_dists <= knn_dists[:, None]
    return float(contained.sum() / (k_eff * M)), float(contained.any(axis=1).mean())

In [5]:
PERTURB_ORDER = ['original', 'simple', 'hard']


def collect_real_fake_vecs(base_key: str, target_key: str, h_tags: list) -> tuple:
    real_rows, fake_rows = [], []
    for p in PERTURB_ORDER:
        base_map = FEATURE_STORE.get(f'{base_key}_{p}_correct', {})
        target_map = FEATURE_STORE.get(f'{target_key}_{p}_correct', {})
        common_probs = set(base_map) & set(target_map)
        for prob_id in common_probs:
            for feat in base_map[prob_id].values():
                real_rows.append(h_vec(feat, h_tags))
            for feat in target_map[prob_id].values():
                fake_rows.append(h_vec(feat, h_tags))
    real_vecs = np.asarray(real_rows, dtype=float) if real_rows else np.empty((0, len(h_tags)), dtype=float)
    fake_vecs = np.asarray(fake_rows, dtype=float) if fake_rows else np.empty((0, len(h_tags)), dtype=float)
    return real_vecs, fake_vecs


rows = []
for base_key, target_key, label, section in PAPER_PAIRS:
    real_vecs, fake_vecs = collect_real_fake_vecs(base_key, target_key, all_h_tags)
    density, coverage = compute_density_coverage(real_vecs, fake_vecs, k=K, metric='cosine')
    rows.append({
        'Section': section,
        'Pair': label,
        'N_base': len(real_vecs),
        'N_PT': len(fake_vecs),
        'Density': round(density, 3) if np.isfinite(density) else float('nan'),
        'Coverage': round(coverage, 3) if np.isfinite(coverage) else float('nan'),
    })

paper_df = pd.DataFrame(rows)
display(paper_df)

,Section,Pair,N_base,N_PT,Density,Coverage
0,Post-trained,Qwen3-1.7B-Base → Qwen3-1.7B-GRPO,834,886,1.220,0.871
1,Post-trained,Olmo-3-7B → Olmo-3-7B-Think-RL-Zero,1229,1307,1.250,0.707
2,Post-trained,Olmo-3-7B → Olmo-3-7B-Think-RLVR,1507,1600,1.032,0.531
3,Cross-model baseline,Olmo-3-7B → Qwen3-1.7B-Base,71,66,0.520,0.437


In [6]:
out_path = OUTPUT_DIR / 'density_coverage_paper_table.csv'
paper_df.to_csv(out_path, index=False)
print('Saved:', out_path)

Saved: /data3/jongsong/SHAPE/analaysis_notebook/outputs/density_coverage_split/density_coverage_paper_table.csv
